<!-- # OFT 训练指标可视化（metrics.csv）

这个 notebook 用于读取 `outputs/<run>/logs/metrics.csv` 并绘制：

- `loss` 曲线（含可选平滑）
- `grad_norm_pre_clip` 曲线
- `step_seconds` / `total_seconds` 耗时曲线
- 多个 run 的曲线对比

> 约定：每个 run 目录下存在 `logs/metrics.csv`，列为：
> `timestamp,global_step,loss,grad_norm_pre_clip,step_seconds,total_seconds` -->


In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# NOTE: run this notebook with working directory at `_OFT/bestcheckpoint`.
OUTPUTS_DIR = Path("../outputs").resolve()

# 显式指定要看的 run（推荐）：
# - 写子目录名（相对 OUTPUTS_DIR），或写绝对路径
# - 只会读取这里列出的 runs，不会扫描整个 outputs/
RUN_DIRS: list[str] = [
    "sd15_oft_run1",
    # "sd35m_std-s_exp",
]

# 如果你暂时想自动发现（会扫描 outputs/），把它设为 True
AUTO_DISCOVER = False

pd.set_option("display.max_rows", 200)
plt.rcParams["figure.dpi"] = 120


In [ ]:
def discover_runs(outputs_dir: Path) -> list[Path]:
    runs: list[Path] = []
    for p in sorted(outputs_dir.iterdir()):
        if not p.is_dir():
            continue
        metrics = p / "logs" / "metrics.csv"
        if metrics.exists():
            runs.append(p)
    return runs


def resolve_run_dirs(outputs_dir: Path, run_dirs: list[str], *, auto_discover: bool) -> list[Path]:
    if run_dirs:
        resolved: list[Path] = []
        for r in run_dirs:
            p = Path(r)
            if not p.is_absolute():
                p = outputs_dir / p
            p = p.resolve()
            metrics = p / "logs" / "metrics.csv"
            if not metrics.exists():
                raise FileNotFoundError(f"metrics.csv not found: {metrics}")
            resolved.append(p)
        return resolved

    if auto_discover:
        runs = discover_runs(outputs_dir)
        if not runs:
            raise FileNotFoundError(f"No runs found under: {outputs_dir}")
        return runs

    raise ValueError("Please set RUN_DIRS (recommended), or set AUTO_DISCOVER=True")


def load_metrics(run_dir: Path) -> pd.DataFrame:
    metrics_path = run_dir / "logs" / "metrics.csv"
    df = pd.read_csv(metrics_path)
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
    # grad_norm_pre_clip 可能是空字符串 -> NaN
    df["grad_norm_pre_clip"] = pd.to_numeric(df["grad_norm_pre_clip"], errors="coerce")
    return df


runs = resolve_run_dirs(OUTPUTS_DIR, RUN_DIRS, auto_discover=AUTO_DISCOVER)
runs


In [ ]:
# 选择要绘制的 run：第 1 个作为主 run，其它用于 overlay 对比
assert runs, "No runs selected"
RUN_DIR = runs[0]

print("Using RUN_DIR:", RUN_DIR)
print("All selected runs:", [p.name for p in runs])

df = load_metrics(RUN_DIR)
df.head()

In [ ]:
def plot_metrics(df: pd.DataFrame, *, smooth_window: int = 1, title: str = ""):
    d = df.sort_values("global_step").copy()

    if smooth_window and smooth_window > 1:
        d["loss_smooth"] = d["loss"].rolling(smooth_window, min_periods=1).mean()
        d["grad_norm_smooth"] = d["grad_norm_pre_clip"].rolling(smooth_window, min_periods=1).mean()
    else:
        d["loss_smooth"] = d["loss"]
        d["grad_norm_smooth"] = d["grad_norm_pre_clip"]

    fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)

    axes[0].plot(d["global_step"], d["loss"], alpha=0.25, label="loss")
    axes[0].plot(d["global_step"], d["loss_smooth"], linewidth=2.0, label=f"loss (smooth={smooth_window})")
    axes[0].set_ylabel("loss")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].plot(d["global_step"], d["grad_norm_pre_clip"], alpha=0.25, label="grad_norm_pre_clip")
    axes[1].plot(d["global_step"], d["grad_norm_smooth"], linewidth=2.0, label=f"grad_norm (smooth={smooth_window})")
    axes[1].set_ylabel("grad_norm")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    axes[2].plot(d["global_step"], d["step_seconds"], label="step_seconds")
    axes[2].set_ylabel("sec/step")
    axes[2].set_xlabel("global_step")
    axes[2].grid(True, alpha=0.3)
    axes[2].legend()

    if title:
        fig.suptitle(title)
    fig.tight_layout()
    return fig


plot_metrics(df, smooth_window=10, title=str(RUN_DIR.name))


In [ ]:
# 多个 run 叠加对比（loss）

def overlay_loss(runs: list[Path], *, smooth_window: int = 10, max_runs: int = 6):
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    for run_dir in runs[:max_runs]:
        d = load_metrics(run_dir).sort_values("global_step")
        y = d["loss"].rolling(smooth_window, min_periods=1).mean() if smooth_window > 1 else d["loss"]
        ax.plot(d["global_step"], y, label=run_dir.name)
    ax.set_xlabel("global_step")
    ax.set_ylabel("loss (smoothed)")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    return fig


overlay_loss(runs, smooth_window=20)


In [ ]:
# 保存图到 figures/（只写到选定的主 run 目录，不写到 outputs 根目录）

FIG_DIR = RUN_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

fig1 = plot_metrics(df, smooth_window=10, title=str(RUN_DIR.name))
fig1_path = FIG_DIR / "metrics_curves.png"
fig1.savefig(fig1_path)
print("Saved:", fig1_path)

fig2 = overlay_loss(runs, smooth_window=20)
fig2_path = FIG_DIR / "overlay_loss.png"
fig2.savefig(fig2_path)
print("Saved:", fig2_path)
